# LoRA hyperparameter sensitivity — training

Trains LoRA adapters with one changed hyperparameter each, seed 42, on the same data as the main runs, and pushes
them to the private model repo `HF_REPO` under `lora-<config>/seed-42/`.

| Config | Change from the main run (r 8, alpha 16, lr 2e-4, linear scheduler, no warmup) |
|---|---|
| `cosine` | cosine scheduler with 3% warmup (Nguyen et al. use cosine) |
| `r32` | rank 32, alpha 64 |
| `lr1e-4` | learning rate 1e-4 |

The main LoRA run with seed 42 (`dimsmauls/qwen2.5-coder-7b-code4ml-lora/seed-42`) is the baseline.

**GPU:** about 31 GB VRAM, so an A100 80 GB (same as the main LoRA runs).
**Before running:** set `hf_token` (write access) and `vast_api_key` in the embedded `config.yml` below (or as
environment variables `HF_TOKEN` / `VAST_API_KEY`), then Run All. A config already on the Hub is skipped, so rerunning
resumes.

## 0. Configuration (edit here)

In [ ]:
%%writefile config.yml
# Credentials. Used only when the matching environment variable (HF_TOKEN / VAST_API_KEY) is not set.
# Keep these empty in any copy you share, commit or upload anywhere except the GPU machine.
hf_token: ""          # Hugging Face token with Write access (huggingface.co/settings/tokens)
vast_api_key: ""      # Vast.ai API key (cloud.vast.ai -> Keys -> API Keys); lets the machine stop itself

auto_shutdown: true    # stop the machine at the end (success or failure); needs VAST_API_KEY

In [ ]:
%pip install -q -U transformers peft accelerate datasets huggingface_hub bitsandbytes

## 1. Configuration

In [ ]:
import gc, json, os, time

import yaml

cfg = yaml.safe_load(open("config.yml"))
# Credentials from config.yml fill in only what the environment does not already provide.
for key, env in (("hf_token", "HF_TOKEN"), ("vast_api_key", "VAST_API_KEY")):
    if cfg.get(key) and not os.environ.get(env):
        os.environ[env] = str(cfg[key])
print("HF_TOKEN set:", bool(os.environ.get("HF_TOKEN")), "| VAST_API_KEY set:", bool(os.environ.get("VAST_API_KEY")))

import torch
from datasets import load_dataset
from huggingface_hub import HfApi, login
from peft import LoraConfig, get_peft_model
from transformers import (AutoModelForCausalLM, AutoTokenizer, DataCollatorForSeq2Seq,
                          EarlyStoppingCallback, Trainer, TrainingArguments, set_seed)

METHOD = "lora"
RUN_CONFIGS = ["cosine", "r32", "lr1e-4"]
SEED = 42

BASE_MODEL = "Qwen/Qwen2.5-Coder-7B-Instruct"
DATASET = "dimsmauls/code4ml-annotated-rated"
DATASET_CONFIG = "training"
HF_REPO = "dimsmauls/qwen2.5-coder-7b-code4ml-hparam"   # private model repo for the sensitivity adapters

# main-run values (Tabel hyperparameter LoRA/QLoRA); each config overrides one of them
BASELINE = dict(r=8, alpha=16, learning_rate=2e-4, lr_scheduler_type="linear", warmup_ratio=0.0)
CONFIGS = {
    "cosine": dict(lr_scheduler_type="cosine", warmup_ratio=0.03),
    "r32": dict(r=32, alpha=64),
    "lr1e-4": dict(learning_rate=1e-4),
}
LORA_DROPOUT = 0.05
TARGET_MODULES = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
EPOCHS = 3
MAX_LENGTH = 2048
PER_DEVICE_BATCH, GRAD_ACCUM, OPTIMIZER = 4, 4, "adamw_bnb_8bit"   # same as the main LoRA runs
BF16 = torch.cuda.is_available() and torch.cuda.is_bf16_supported()

import peft, transformers
print(torch.cuda.get_device_name(0), "| bf16:", BF16, "| configs:", RUN_CONFIGS)
print("transformers", transformers.__version__, "| peft", peft.__version__, "| torch", torch.__version__)

## 2. Log in to Hugging Face

In [ ]:
token = os.environ.get("HF_TOKEN")
login(token=token) if token else login()
api = HfApi()
api.create_repo(HF_REPO, private=True, exist_ok=True)
print("pushing adapters to", HF_REPO)

## 3. Load and format the dataset
Same as the main training notebooks: chat format, loss only on the code tokens.

In [ ]:
data = load_dataset(DATASET, DATASET_CONFIG, split="train")
train_rows = data.filter(lambda row: row["split"] == "train")
val_rows = data.filter(lambda row: row["split"] == "validation")
print("train:", len(train_rows), "| validation:", len(val_rows))

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL)

def encode(row):
    """Chat format: user = instruction, assistant = code. Loss only on the code tokens."""
    prompt = [{"role": "user", "content": row["instruction"]}]
    full = prompt + [{"role": "assistant", "content": row["code"]}]
    prompt_text = tokenizer.apply_chat_template(prompt, add_generation_prompt=True, tokenize=False)
    full_text = tokenizer.apply_chat_template(full, tokenize=False)
    prompt_ids = tokenizer(prompt_text, add_special_tokens=False)["input_ids"]
    full_ids = tokenizer(full_text, add_special_tokens=False)["input_ids"][:MAX_LENGTH]
    labels = [-100] * len(prompt_ids) + full_ids[len(prompt_ids):]
    return {"input_ids": full_ids, "attention_mask": [1] * len(full_ids), "labels": labels[:len(full_ids)]}

columns = train_rows.column_names
train_ds = train_rows.map(encode, remove_columns=columns)
val_ds = val_rows.map(encode, remove_columns=columns)
collator = DataCollatorForSeq2Seq(tokenizer, padding=True, label_pad_token_id=-100)

## 4. Training function for one config
Same procedure as the main LoRA runs (early stopping on validation loss, best epoch kept). The adapter and
`metrics.json` (including the hyperparameters used) go to `HF_REPO/lora-<config>/seed-42/`.

In [ ]:
# def train_one(name: str) -> dict:
#     hp = {**BASELINE, **CONFIGS[name]}
#     set_seed(SEED)
#     torch.cuda.reset_peak_memory_stats()
#     start = time.time()

#     model = AutoModelForCausalLM.from_pretrained(
#         BASE_MODEL, torch_dtype=torch.bfloat16 if BF16 else torch.float16, device_map="auto")
#     model.gradient_checkpointing_enable()
#     model.enable_input_require_grads()
#     model = get_peft_model(model, LoraConfig(r=hp["r"], lora_alpha=hp["alpha"], lora_dropout=LORA_DROPOUT,
#                                              target_modules=TARGET_MODULES, task_type="CAUSAL_LM"))
#     model.print_trainable_parameters()

#     out_dir = f"outputs/{METHOD}-{name}-seed-{SEED}"
#     args = TrainingArguments(
#         output_dir=out_dir, seed=SEED, data_seed=SEED,
#         # num_train_epochs=EPOCHS, learning_rate=hp["learning_rate"],
#         # lr_scheduler_type=hp["lr_scheduler_type"], warmup_ratio=hp["warmup_ratio"],
#         num_train_epochs=EPOCHS, learning_rate=hp["learning_rate"],
#         lr_scheduler_type=hp["lr_scheduler_type"],
#         warmup_steps=hp["warmup_ratio"],
#         per_device_train_batch_size=PER_DEVICE_BATCH, per_device_eval_batch_size=PER_DEVICE_BATCH,
#         gradient_accumulation_steps=GRAD_ACCUM, optim=OPTIMIZER,
#         bf16=BF16, fp16=not BF16, gradient_checkpointing=True,
#         eval_strategy="epoch", save_strategy="epoch", save_total_limit=1,
#         load_best_model_at_end=True, metric_for_best_model="eval_loss", greater_is_better=False,
#         logging_steps=10, report_to="none",
#     )
#     trainer = Trainer(model=model, args=args, train_dataset=train_ds, eval_dataset=val_ds,
#                       data_collator=collator, callbacks=[EarlyStoppingCallback(early_stopping_patience=1)])
#     trainer.train()

#     adapter_dir = f"{out_dir}/adapter"
#     trainer.model.save_pretrained(adapter_dir)
#     tokenizer.save_pretrained(adapter_dir)
#     metrics = {
#         "method": METHOD, "config": name, "seed": SEED, "base_model": BASE_MODEL, "hyperparameters": hp,
#         "per_device_batch": PER_DEVICE_BATCH, "grad_accum": GRAD_ACCUM, "optimizer": OPTIMIZER,
#         "train_samples": len(train_ds), "validation_samples": len(val_ds),
#         "best_eval_loss": trainer.state.best_metric, "epochs_run": trainer.state.epoch,
#         "train_minutes": round((time.time() - start) / 60, 1),
#         "peak_vram_gb": round(torch.cuda.max_memory_allocated() / 1024**3, 2),
#         "gpu": torch.cuda.get_device_name(0),
#     }
#     with open(f"{adapter_dir}/metrics.json", "w") as handle:
#         json.dump(metrics, handle, indent=2)

#     api.upload_folder(repo_id=HF_REPO, folder_path=adapter_dir, path_in_repo=f"{METHOD}-{name}/seed-{SEED}",
#                       commit_message=f"{METHOD}-{name} adapter seed {SEED}")
#     print(json.dumps(metrics, indent=2))

#     del trainer, model
#     gc.collect()
#     torch.cuda.empty_cache()
#     return metrics

def train_one(name: str) -> dict:
    hp = {**BASELINE, **CONFIGS[name]}
    set_seed(SEED)
    torch.cuda.reset_peak_memory_stats()
    start = time.time()

    model = AutoModelForCausalLM.from_pretrained(
        BASE_MODEL,
        dtype=torch.bfloat16 if BF16 else torch.float16,
        device_map="auto",
    )
    model.gradient_checkpointing_enable()
    model.enable_input_require_grads()

    model = get_peft_model(
        model,
        LoraConfig(
            r=hp["r"],
            lora_alpha=hp["alpha"],
            lora_dropout=LORA_DROPOUT,
            target_modules=TARGET_MODULES,
            task_type="CAUSAL_LM",
        ),
    )
    model.print_trainable_parameters()

    out_dir = f"outputs/{METHOD}-{name}-seed-{SEED}"

    args = TrainingArguments(
        output_dir=out_dir,
        seed=SEED,
        data_seed=SEED,
        num_train_epochs=EPOCHS,
        learning_rate=hp["learning_rate"],
        lr_scheduler_type=hp["lr_scheduler_type"],
        warmup_steps=hp["warmup_ratio"],
        per_device_train_batch_size=PER_DEVICE_BATCH,
        per_device_eval_batch_size=PER_DEVICE_BATCH,
        gradient_accumulation_steps=GRAD_ACCUM,
        optim=OPTIMIZER,
        bf16=BF16,
        fp16=not BF16,
        gradient_checkpointing=True,
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=1,
        load_best_model_at_end=True,
        metric_for_best_model="eval_loss",
        greater_is_better=False,
        logging_steps=10,
        report_to="none",
    )

    trainer = Trainer(
        model=model,
        args=args,
        train_dataset=train_ds,
        eval_dataset=val_ds,
        data_collator=collator,
        callbacks=[
            EarlyStoppingCallback(early_stopping_patience=1)
        ],
    )
    trainer.train()

    adapter_dir = f"{out_dir}/adapter"
    trainer.model.save_pretrained(adapter_dir)
    tokenizer.save_pretrained(adapter_dir)

    metrics = {
        "method": METHOD,
        "config": name,
        "seed": SEED,
        "base_model": BASE_MODEL,
        "hyperparameters": hp,
        "per_device_batch": PER_DEVICE_BATCH,
        "grad_accum": GRAD_ACCUM,
        "optimizer": OPTIMIZER,
        "train_samples": len(train_ds),
        "validation_samples": len(val_ds),
        "best_eval_loss": trainer.state.best_metric,
        "epochs_run": trainer.state.epoch,
        "train_minutes": round((time.time() - start) / 60, 1),
        "peak_vram_gb": round(
            torch.cuda.max_memory_allocated() / 1024**3, 2
        ),
        "gpu": torch.cuda.get_device_name(0),
    }

    with open(f"{adapter_dir}/metrics.json", "w") as handle:
        json.dump(metrics, handle, indent=2)

    api.upload_folder(
        repo_id=HF_REPO,
        folder_path=adapter_dir,
        path_in_repo=f"{METHOD}-{name}/seed-{SEED}",
        commit_message=f"{METHOD}-{name} adapter seed {SEED}",
    )

    print(json.dumps(metrics, indent=2))

    del trainer, model
    gc.collect()
    torch.cuda.empty_cache()

    return metrics

## 5. Run the configs
A config whose adapter is already on the Hub is skipped. A failure uploads its traceback to `HF_REPO/errors/`
and the shutdown cell still runs.

In [ ]:
import traceback
from datetime import datetime, timezone

results, run_error = [], None
on_hub = set(api.list_repo_files(HF_REPO))
try:
    for name in RUN_CONFIGS:
        if f"{METHOD}-{name}/seed-{SEED}/adapter_model.safetensors" in on_hub:
            print(f"{METHOD}-{name}: already on the Hub, skipping")
            continue
        print(f"===== {METHOD}-{name} seed {SEED} =====")
        results.append(train_one(name))
except Exception:
    run_error = traceback.format_exc()
    print(run_error)
    stamp = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
    try:
        api.upload_file(path_or_fileobj=run_error.encode(), path_in_repo=f"errors/{METHOD}-{stamp}.txt",
                        repo_id=HF_REPO, commit_message=f"{METHOD} sensitivity training error {stamp}")
    except Exception as upload_exc:
        print("could not upload the error log:", upload_exc)

print(json.dumps(results, indent=2))

## 6. Shut the machine down
Vast.ai: `vastai stop instance` (needs `VAST_API_KEY`; `CONTAINER_ID` is set by Vast.ai). Colab: releases the runtime.

In [ ]:
def shutdown() -> None:
    try:
        from google.colab import runtime  # only importable on Colab
        runtime.unassign()
        return
    except ImportError:
        pass
    import subprocess
    container_id, api_key = os.environ.get("CONTAINER_ID"), os.environ.get("VAST_API_KEY")
    if not (container_id and api_key):
        print("CONTAINER_ID or VAST_API_KEY not set: STOP THE INSTANCE YOURSELF on cloud.vast.ai")
        return
    subprocess.run(["pip", "install", "-q", "vastai"], check=True)
    print(f"stopping Vast.ai instance {container_id}")
    subprocess.run(["vastai", "stop", "instance", container_id, "--api-key", api_key], check=True)

print("failed:" if run_error else "done:", [f"{METHOD}-{m['config']}" for m in results])
if cfg.get("auto_shutdown", True):
    shutdown()
else:
    print("auto_shutdown is off: stop the machine yourself")